In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluate import evaluate
from features import GROUP, TARGET, load_rated, make_splits
from models import fit_predict_lgbm, fit_predict_logreg

rated = load_rated()
train, valid, test = make_splits(rated)
dev = pd.concat([train, valid]).reset_index(drop=True)     # test stays locked
strata = dev[GROUP] + "_" + dev[TARGET].astype(str)
print(f"Development set: {len(dev):,} businesses")


def fold_scores(fit_predict, seed, **params):
    """Train on 4 folds, score the 5th, five times. Returns one metrics row per fold."""
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    rows = []
    for fold, (fit_idx, score_idx) in enumerate(folds.split(dev, strata), start=1):
        fit_part, score_part = dev.iloc[fit_idx], dev.iloc[score_idx]
        row = evaluate("", score_part, fit_predict(fit_part, score_part, **params))
        row["fold"] = fold
        rows.append(row)
    return pd.DataFrame(rows).drop(columns="model")

c:\projects\food-hygiene-risk-london\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Development set: 57,112 businesses


In [2]:
TUNING_SEED = 42      # folds used to choose settings


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 64, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 400, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
    }
    return fold_scores(fit_predict_lgbm, TUNING_SEED, **params)["recall@20%"].mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"\nBest mean recall@20% on tuning folds: {study.best_value:.3f}")
print("Best settings:")
for name, value in study.best_params.items():
    print(f"   {name:<18} {value}")

Best trial: 17. Best value: 0.435566: 100%|██████████| 40/40 [05:46<00:00,  8.65s/it]


Best mean recall@20% on tuning folds: 0.436
Best settings:
   n_estimators       250
   learning_rate      0.016559893380029844
   num_leaves         27
   min_child_samples  69
   subsample          0.5233873934557627
   colsample_bytree   0.6579054698627168
   reg_lambda         0.005145119158889496
   class_weight       None


In [3]:
CHECK_SEED = 7        # brand-new folds, never used for tuning

comparison = []
for name, fit_predict, params in [
    ("Logistic regression", fit_predict_logreg, {}),
    ("LightGBM (default)", fit_predict_lgbm, {}),
    ("LightGBM (tuned)", fit_predict_lgbm, study.best_params),
]:
    scores = fold_scores(fit_predict, CHECK_SEED, **params)
    scores["model"] = name
    comparison.append(scores)
    print(f"{name} done")

comparison = pd.concat(comparison)
metrics = ["recall@20%", "recall@10%", "PR-AUC (within)"]
print("\nMean and std on NEW folds:\n")
print(comparison.groupby("model")[metrics].agg(["mean", "std"]).round(3).to_string())

per_fold = comparison.pivot(index="fold", columns="model", values="recall@20%")
per_fold["Tuned minus LogReg"] = per_fold["LightGBM (tuned)"] - per_fold["Logistic regression"]
print("\nrecall@20% per fold:\n")
print(per_fold.round(3).to_string())

wins = int((per_fold["Tuned minus LogReg"] > 0).sum())
print(f"\nTuned LightGBM beats logistic regression on {wins} of 5 folds.")
print("Decision rule (at least 4 of 5):", "LightGBM WINS" if wins >= 4 else "keep LOGISTIC REGRESSION")

Logistic regression done
LightGBM (default) done
LightGBM (tuned) done

Mean and std on NEW folds:

                    recall@20%        recall@10%        PR-AUC (within)       
                          mean    std       mean    std            mean    std
model                                                                         
LightGBM (default)       0.407  0.018      0.236  0.010           0.145  0.010
LightGBM (tuned)         0.419  0.027      0.251  0.006           0.146  0.008
Logistic regression      0.415  0.018      0.235  0.014           0.133  0.014

recall@20% per fold:

model  LightGBM (default)  LightGBM (tuned)  Logistic regression  Tuned minus LogReg
fold                                                                                
1                   0.404             0.408                0.422              -0.014
2                   0.431             0.443                0.401               0.042
3                   0.402             0.424                0.42

In [4]:
params_dir = PROJECT_ROOT / "models"
params_dir.mkdir(exist_ok=True)
with open(params_dir / "lgbm_best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)
print("Saved to", params_dir / "lgbm_best_params.json")

Saved to c:\projects\food-hygiene-risk-london\models\lgbm_best_params.json


## Tuning result

- Optuna, 40 trials, objective = mean recall@20% on tuning folds (seed 42).
- Fair comparison on NEW folds (seed 7) to avoid the winner's curse:

| Model | recall@20% | recall@10% | PR-AUC (within) |
|---|---|---|---|
| Logistic regression | 0.415 ± 0.018 | 0.235 ± 0.014 | 0.133 ± 0.014 |
| LightGBM (default) | 0.407 ± 0.018 | 0.236 ± 0.010 | 0.145 ± 0.010 |
| LightGBM (tuned) | 0.419 ± 0.027 | 0.251 ± 0.006 | 0.146 ± 0.008 |

- **Decision rule (set before tuning): tuned LightGBM needed to beat logistic
  regression on at least 4 of 5 folds. It won 2 of 5, so logistic regression is the
  final model.**
- Tuning added only +0.012 recall@20% over LightGBM defaults. A linear and a tree
  model reach the same ceiling (about 41%), so the limit is the information in the
  features, not the algorithm. Future gains should come from better data
  (e.g. inspection history from repeated snapshots).
- Observation for future work: LightGBM shows a small, consistent edge at the very
  top of the list (recall@10% 0.251 vs 0.235; higher PR-AUC within in both CV runs).
  Worth a pre-registered test if councils' real capacity is about 10%.